In [141]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### data loading

In [142]:

df_arrests: pd.DataFrame = pd.read_csv('/Users/leahmayotte/Documents/education/umass/capstone/2_data/raw/arrests-latest.csv', low_memory=False)
df_detainers: pd.DataFrame  = pd.read_csv('/Users/leahmayotte/Documents/education/umass/capstone/2_data/raw/detainers-latest.csv', low_memory=False)
df_detention_stays: pd.DataFrame = pd.read_csv('/Users/leahmayotte/Documents/education/umass/capstone/2_data/raw/detention_stays-latest.csv', low_memory=False)

print(f"    - {len(df_arrests)} arrests")
print(f"    - {len(df_detainers)} detainers")
print(f"    - {len(df_detention_stays)} detentions")

    - 377067 arrests
    - 396306 detainers
    - 671750 detentions


### data exploration

In [143]:
df_arrests.info()

<class 'pandas.DataFrame'>
RangeIndex: 377067 entries, 0 to 377066
Data columns (total 23 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   apprehension_date           377067 non-null  str    
 1   apprehension_state          316913 non-null  str    
 2   apprehension_aor            371180 non-null  str    
 3   final_program               377067 non-null  str    
 4   final_program_group         377067 non-null  str    
 5   apprehension_method         377067 non-null  str    
 6   apprehension_criminality    377067 non-null  str    
 7   case_status                 367695 non-null  str    
 8   case_category               367694 non-null  str    
 9   departed_date               236138 non-null  str    
 10  departure_country           236084 non-null  str    
 11  final_order_yes_no          367695 non-null  str    
 12  final_order_date            229195 non-null  str    
 13  birth_year               

In [144]:
df_detainers.info()

<class 'pandas.DataFrame'>
RangeIndex: 396306 entries, 0 to 396305
Data columns (total 62 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   detainer_prepare_date           396306 non-null  str    
 1   facility_state                  395876 non-null  str    
 2   facility_aor                    317180 non-null  str    
 3   port_of_departure               103412 non-null  str    
 4   departure_country               103412 non-null  str    
 5   departed_date                   103493 non-null  str    
 6   case_status                     193347 non-null  str    
 7   detainer_prepared_criminality   396306 non-null  str    
 8   detention_facility              396290 non-null  str    
 9   detention_facility_code         396290 non-null  str    
 10  facility_city                   395876 non-null  str    
 11  detainer_prep_threat_level      374076 non-null  float64
 12  gender                     

In [145]:
df_detention_stays.info()

<class 'pandas.DataFrame'>
RangeIndex: 671750 entries, 0 to 671749
Data columns (total 45 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   stay_ID                          671750 non-null  str    
 1   n_stints                         671750 non-null  int64  
 2   detention_facility_codes_all     671750 non-null  str    
 3   stay_book_in_date_time           671750 non-null  str    
 4   stay_book_out_date_time          609849 non-null  str    
 5   detention_release_reason         610132 non-null  str    
 6   stay_book_out_date               609849 non-null  str    
 7   stay_release_reason              609851 non-null  str    
 8   religion                         14041 non-null   str    
 9   gender                           671750 non-null  str    
 10  marital_status                   520629 non-null  str    
 11  birth_year                       671750 non-null  int64  
 12  ethnicity    

### null handling

In [146]:
def calculate_null_percentage():

    arrests_null_percentages: float = (df_arrests.isnull().sum() / len(df_arrests) * 100).round(2)
    print("Arrests Table Null Percentages")
    with pd.option_context('display.max_rows', None):
        print(arrests_null_percentages)
    print()
    print()

    detainers_null_percentages = (df_detainers.isnull().sum() / len(df_detainers) * 100).round(2)
    print("Detainers Table Null Percentages")
    with pd.option_context('display.max_rows', None):
        print(detainers_null_percentages)
    print()
    print()

    detentions_null_percentages = (df_detention_stays.isnull().sum() / len(df_detention_stays) * 100).round(2)
    print("Detention Stays Table Null Percentages")
    with pd.option_context('display.max_rows', None):
        print(detentions_null_percentages)

calculate_null_percentage()

Arrests Table Null Percentages
apprehension_date              0.00
apprehension_state            15.95
apprehension_aor               1.56
final_program                  0.00
final_program_group            0.00
apprehension_method            0.00
apprehension_criminality       0.00
case_status                    2.49
case_category                  2.49
departed_date                 37.38
departure_country             37.39
final_order_yes_no             2.49
final_order_date              39.22
birth_year                     0.00
citizenship_country            0.00
gender                         0.00
apprehension_site_landmark     2.33
unique_identifier              1.43
apprehension_date_time         0.00
duplicate_likely               1.43
file_original                  0.00
sheet_original                 0.00
row_original                   0.00
dtype: float64


Detainers Table Null Percentages
detainer_prepare_date               0.00
facility_state                      0.11
facility_

In [147]:
# find outlier Null values in each table
print('Scanning for Suspicious Null Values...')

# create list of values that I want to look for
suspicious_null_values: list[str] = ['','-', 'N/A', 'None', 'NULL', 'null', 'none', 'unknown', '--']
# only return column names that are strings
arrests_string_columns: pd.Index = df_arrests.select_dtypes(include='str').columns
detainers_string_columns: pd.Index = df_detainers.select_dtypes(include='str').columns
detentions_string_columns: pd.Index = df_detention_stays.select_dtypes(include='str').columns

# iterate over each column
for column in arrests_string_columns:
    # look for only unique values, dropping NaN values, so only non-null values are inspected
    unique_vals: np.ndarray = df_arrests[column].dropna().unique()
    # as the loop iterates, build a list of values that match the suspicious_null_values defined
    suspicious: list[str] = []
    for value in unique_vals:
        if str(value).strip() in suspicious_null_values:
            suspicious.append(value)
    if suspicious:
        print('Arrests Table: Suspicious Nulls Found')
        print(f'{column}: {suspicious}')

print('Arrests Table: Scan Complete')

for column in detainers_string_columns:
    unique_vals: np.ndarray = df_detainers[column].dropna().unique()
    suspicious: list[str] = []
    for value in unique_vals:
        if str(value).strip() in suspicious_null_values:
             suspicious.append(value)
    if suspicious:
        print('Detainers Table: Suspicious Nulls Found')
        print(f'{column}: {suspicious}')

print('Detainers Table: Scan Complete')

for column in detentions_string_columns:
    unique_vals: np.ndarray = df_detention_stays[column].dropna().unique()
    suspicious: list[str] = []
    for value in unique_vals:
        if str(value).strip() in suspicious_null_values:
            suspicious.append(value)
    if suspicious:
        print('Detention Stays Table: Suspicious Nulls Found')
        print(f'{column}: {suspicious}')

print('Detention Stays Table: Scan Complete')


Scanning for Suspicious Null Values...
Arrests Table: Scan Complete
Detainers Table: Scan Complete
Detention Stays Table: Suspicious Nulls Found
religion: ['unknown', 'none']
Detention Stays Table: Scan Complete


### removing unnecessary data

In [148]:
# check for existing columns
print(df_arrests.columns.tolist())
print()
print(df_detainers.columns.tolist())
print()
print(df_detention_stays.columns.tolist())

['apprehension_date', 'apprehension_state', 'apprehension_aor', 'final_program', 'final_program_group', 'apprehension_method', 'apprehension_criminality', 'case_status', 'case_category', 'departed_date', 'departure_country', 'final_order_yes_no', 'final_order_date', 'birth_year', 'citizenship_country', 'gender', 'apprehension_site_landmark', 'unique_identifier', 'apprehension_date_time', 'duplicate_likely', 'file_original', 'sheet_original', 'row_original']

['detainer_prepare_date', 'facility_state', 'facility_aor', 'port_of_departure', 'departure_country', 'departed_date', 'case_status', 'detainer_prepared_criminality', 'detention_facility', 'detention_facility_code', 'facility_city', 'detainer_prep_threat_level', 'gender', 'citizenship_country', 'birth_country', 'birth_year', 'entry_status', 'most_serious_conviction_charge', 'msc_sentence_days', 'msc_sentence_months', 'msc_sentence_years', 'msc_charge_code', 'msc_charge_date', 'msc_conviction_date', 'felon', 'processing_disposition'

In [149]:
# create lists naming the columns to be removed
arrests_cols_to_drop: list[str] = [
    # 100% the same value
    'file_original',
    'sheet_original',
    'final_program_group',
]

detainers_cols_to_drop: list[str] = [
    # 100% same value
    'file_original',
    'sheet_original',
    # 100% null
    'unlawful_attempt_yes_no',
    'unlawful_entry_yes_no',
    'federal_interest_yes_no',
    'visa_yes_no',
    # effectively null
    'illegal_entry_yes_no',
    'immigration_fraud_yes_no',
    'other_removal_reason_yes_no',
    # not relevant to analysis
    'port_of_departure',
    'arrest_time_case_category',
    'arrest_time_current_program',
    'msc_charge_date',
    'msc_conviction_date',
    'order_show_cause_served_yes_no',
    'statements_made_yes_no',
]

detention_stays_cols_to_drop: list[str] = [
    # effectively null
    'religion',
    # redundant or not relevant
    'stay_book_out_date',
    'detention_facility_codes_all',
    'book_in_date_time_first',
]

# drop columns specified above
df_arrests = df_arrests.drop(columns=arrests_cols_to_drop)
df_detainers = df_detainers.drop(columns=detainers_cols_to_drop)
df_detention_stays = df_detention_stays.drop(columns=detention_stays_cols_to_drop)

### duplicate handling

In [150]:
# utilize the existing duplicate_likely column as a flag for duplicates
df_arrests = df_arrests[df_arrests['duplicate_likely'] == 0]
df_detainers = df_detainers[df_detainers['duplicate_likely'] == 0]

# once filtered, drop the 'duplicate_likely' column as it is no longer necessary
df_arrests = df_arrests.drop(columns=['duplicate_likely'])
df_detainers = df_detainers.drop(columns=['duplicate_likely'])

In [151]:
# recheck columns for each table
print(df_arrests.columns.tolist())
print()
print(df_detainers.columns.tolist())
print()
print(df_detention_stays.columns.tolist())

['apprehension_date', 'apprehension_state', 'apprehension_aor', 'final_program', 'apprehension_method', 'apprehension_criminality', 'case_status', 'case_category', 'departed_date', 'departure_country', 'final_order_yes_no', 'final_order_date', 'birth_year', 'citizenship_country', 'gender', 'apprehension_site_landmark', 'unique_identifier', 'apprehension_date_time', 'row_original']

['detainer_prepare_date', 'facility_state', 'facility_aor', 'departure_country', 'departed_date', 'case_status', 'detainer_prepared_criminality', 'detention_facility', 'detention_facility_code', 'facility_city', 'detainer_prep_threat_level', 'gender', 'citizenship_country', 'birth_country', 'birth_year', 'entry_status', 'most_serious_conviction_charge', 'msc_sentence_days', 'msc_sentence_months', 'msc_sentence_years', 'msc_charge_code', 'felon', 'processing_disposition', 'case_category', 'final_program', 'apprehension_method', 'case_final_order_yes_no', 'final_order_date', 'apprehension_date', 'entry_date', 

### data type correction

In [152]:
# check column data types
df_arrests.dtypes

apprehension_date                 str
apprehension_state                str
apprehension_aor                  str
final_program                     str
apprehension_method               str
apprehension_criminality          str
case_status                       str
case_category                     str
departed_date                     str
departure_country                 str
final_order_yes_no                str
final_order_date                  str
birth_year                    float64
citizenship_country               str
gender                            str
apprehension_site_landmark        str
unique_identifier                 str
apprehension_date_time            str
row_original                    int64
dtype: object

In [153]:
df_detainers.dtypes

detainer_prepare_date                 str
facility_state                        str
facility_aor                          str
departure_country                     str
departed_date                         str
case_status                           str
detainer_prepared_criminality         str
detention_facility                    str
detention_facility_code               str
facility_city                         str
detainer_prep_threat_level        float64
gender                                str
citizenship_country                   str
birth_country                         str
birth_year                        float64
entry_status                          str
most_serious_conviction_charge        str
msc_sentence_days                 float64
msc_sentence_months               float64
msc_sentence_years                float64
msc_charge_code                       str
felon                                 str
processing_disposition                str
case_category                     

In [154]:
df_detention_stays.dtypes

stay_ID                                str
n_stints                             int64
stay_book_in_date_time                 str
stay_book_out_date_time                str
detention_release_reason               str
stay_release_reason                    str
gender                                 str
marital_status                         str
birth_year                           int64
ethnicity                              str
entry_status                           str
felon                                  str
bond_posted_date                       str
bond_posted_amount                 float64
case_status                            str
case_category                          str
final_order_yes_no                     str
final_order_date                       str
case_threat_level                  float64
book_in_criminality                    str
final_charge                           str
departed_date                          str
departure_country                      str
initial_bon

In [155]:
# convert to DATETIME

arrests_date_cols: list[str] = [
    'apprehension_date',
    'apprehension_date_time',
    'departed_date',
]

detainers_date_cols: list[str] = [
    'detainer_prepare_date',
    'final_order_date',
    'apprehension_date',
    'entry_date',
    'departed_date',
]

detention_stays_date_cols: list[str] = [
    'stay_book_in_date_time',
    'stay_book_out_date_time',
    'final_order_date',
    'bond_posted_date',
    'book_out_date_time_first',
    'book_in_date_time_longest',
    'book_out_date_time_longest',
    'book_in_date_time_last',
    'book_out_date_time_last',
]

for col in arrests_date_cols:
    df_arrests[col] = pd.to_datetime(df_arrests[col])

for col in detainers_date_cols:
    df_detainers[col] = pd.to_datetime(df_detainers[col])

for col in detention_stays_date_cols:
    df_detention_stays[col] = pd.to_datetime(df_detention_stays[col])


In [156]:
# convert birth year to integer (preserve nulls)
df_arrests['birth_year'] = df_arrests['birth_year'].astype('Int64')
df_detainers['birth_year'] = df_detainers['birth_year'].astype('Int64')

In [157]:
# standardize text columns
arrests_title_cols: list[str] = [
    'apprehension_state',
    'apprehension_aor',
    'apprehension_method',
    'apprehension_criminality',
    'final_program',
    'case_status',
    'case_category',
    'departure_country',
    'citizenship_country',
    'gender',
]

detainer_title_cols: list[str] = [
    'facility_state',
    'facility_aor',
    'departure_country',
    'case_status',
    'detainer_prepared_criminality',
    'detention_facility',
    'gender',
    'citizenship_country',
    'birth_country',
    'entry_status',
    'most_serious_conviction_charge',
    'felon',
    'processing_disposition',
    'case_category',
    'final_program',
    'detainer_type',
    'detainer_lift_reason',
]

detention_stays_title_cols: list[str] = [
    'gender',
    'citizenship_country',
    'ethnicity',
    'entry_status',
    'felon',
    'case_status',
    'case_category',
    'book_in_criminality',
    'final_charge',
    'departure_country',
    'final_program',
    'detention_release_reason',
    'stay_release_reason',
    'marital_status',
    'msc_charge',
    'detention_facility_first',
    'detention_facility_longest',
    'detention_facility_last',
]

for col in arrests_title_cols:
    df_arrests[col] = df_arrests[col].str.strip().str.upper()

for col in detainer_title_cols:
    df_detainers[col] = df_detainers[col].str.strip().str.upper()

for col in detention_stays_title_cols:
    df_detention_stays[col] = df_detention_stays[col].str.strip().str.upper()

In [158]:
# # verify text change
# df_arrests.head()

In [159]:
# df_detainers.head()

In [160]:
# df_detention_stays.head()

In [161]:
# convert appropriate binaries to booleans

arrests_bool_cols: list[str] = [
    'final_order_yes_no'
]

detainers_bool_cols: list[str] = [
    'resume_custody_yes_no',
    'biometric_match_yes_no',
    'prior_felony_yes_no',
    'violent_misdemeanor_yes_no',
    'aggravated_felony_yes_no',
    'deportation_ordered_yes_no',
    'final_order_yes_no',
    'case_final_order_yes_no',
]

detention_stays_bool_cols: list[str] = [
    'final_order_yes_no',
]

# recognize YES/NO as TRUE/FALSE
bool_type: dict[str, bool] = {
    'YES' : True,
    'NO' : False
}

for col in arrests_bool_cols:
    df_arrests[col] = df_arrests[col].map(bool_type).astype('boolean')

for col in detainers_bool_cols:
    df_detainers[col] = df_detainers[col].map(bool_type).astype('boolean')

for col in detention_stays_bool_cols:
    df_detention_stays[col] = df_detention_stays[col].map(bool_type).astype('boolean')

# check that it worked
print(df_arrests['final_order_yes_no'].value_counts(dropna=False))
print(df_arrests['final_order_yes_no'].dtype)


final_order_yes_no
True     225003
False    130333
<NA>       6866
Name: count, dtype: Int64
boolean


### data creation for analysis

In [162]:
# simplifying criminality data for analysis
criminality_type: dict[str, str] = {
    '1 Convicted Criminal' :    'Convicted Criminal',
    '2 Pending Criminal Charges':   'Pending Charges',
    '3 Other Immigration Violator':    'Other Immigration Violator'
}

df_arrests['criminality_clean'] = df_arrests['apprehension_criminality'].map(criminality_type)
df_detainers['criminality_clean'] = df_detainers['detainer_prepared_criminality'].map(criminality_type)
df_detention_stays['criminality_clean'] = df_detention_stays['book_in_criminality'].map(criminality_type)

In [163]:
# calculate detention length in detention_stays

df_detention_stays['detention_length_days'] = (
    df_detention_stays['stay_book_out_date_time'] -
    df_detention_stays['stay_book_in_date_time']
).dt.days

In [164]:
# create an 'age' column using 'birth_year' conversion

from datetime import datetime

current_year: int = datetime.now().year

df_arrests['age'] = current_year - df_arrests['birth_year']
df_detainers['age'] = current_year - df_detainers['birth_year']
df_detention_stays['age'] = current_year -df_detention_stays['birth_year']

In [165]:
# extract year and month from critical date columns
df_arrests['apprehension_year'] = df_arrests['apprehension_date'].dt.year
df_arrests['apprehension_month'] = df_arrests['apprehension_date'].dt.month

df_detainers['detainer_year'] = df_detainers['detainer_prepare_date'].dt.year
df_detainers['detainer_month'] = df_detainers['detainer_prepare_date'].dt.month

df_detention_stays['stay_year'] = df_detention_stays['stay_book_in_date_time'].dt.year
df_detention_stays['stay_month'] = df_detention_stays['stay_book_in_date_time'].dt.month

In [166]:
# convert sentence to days for comparative duration columns

df_detainers['msc_sentence_days_total'] = (
    df_detainers['msc_sentence_days'].fillna(0) +
    (df_detainers['msc_sentence_months'].fillna(0) * 30.44) +
    (df_detainers['msc_sentence_years'].fillna(0) * 365)
).round(0)

# Where all three original columns are null, set the total to NaN
all_null_mask = (
    df_detainers['msc_sentence_days'].isna() &
    df_detainers['msc_sentence_months'].isna() &
    df_detainers['msc_sentence_years'].isna()
)

df_detainers.loc[all_null_mask, 'msc_sentence_days_total'] = np.nan
print(f'Rows that were null:    {all_null_mask.sum():,}')
print(f'Rows null after mask:    {df_detainers['msc_sentence_days_total'].isna().sum():,}')

# Where all three original columns are zero, set the total to NaN
all_zeros_mask = (
    (df_detainers['msc_sentence_days'] == 0) &
    (df_detainers['msc_sentence_months'] == 0) &
    (df_detainers['msc_sentence_years'] == 0)
)

df_detainers.loc[all_zero_mask, 'msc_sentence_days_total'] = np.nan
print(f"Rows that were zeros:  {all_zeros_mask.sum():,}")
print(f"Rows zeros after mask:      {df_detainers['msc_sentence_days_total'].isna().sum():,}")
# Total nulls expected after setting NaN values
print(f"Expected total nulls:        {all_null_mask.sum() + all_zeros_mask.sum():,}")

# check the distribution of the new 'msc_sentence_days_total' column
print(df_detainers['msc_sentence_days_total'].describe())


Rows that were null:    22,854
Rows null after mask:    22,854
Rows that were zeros:  228,111
Rows zeros after mask:      250,965
Expected total nulls:        250,965
count     78518.000000
mean       1178.954953
std        3915.780064
min           0.000000
25%          68.000000
50%         365.000000
75%        1097.000000
max      364635.000000
Name: msc_sentence_days_total, dtype: float64


### data export

In [167]:
cleaned_data_path: str = '/Users/leahmayotte/Documents/education/umass/capstone/2_data/cleaned/'

df_arrests.to_csv(f'{cleaned_data_path}arrests_cleaned.csv', index=False)
df_detainers.to_csv(f'{cleaned_data_path}detainers_cleaned.csv', index=False)
df_detention_stays.to_csv(f'{cleaned_data_path}detention_stays_cleaned.csv', index=False)